In [20]:
import sys
import os

# 让 notebook 能 import ml-service 根目录下的模块
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
from utils.preprocessor import preprocess_to_string
from config import CORPUS_FILE

# 加载原始语料
df = pd.read_csv(str(CORPUS_FILE), encoding="utf-8-sig")
print(f"原始数据: {len(df)} 条")
print(f"列名: {list(df.columns)}")
df.head()

原始数据: 7728 条
列名: ['id', 'content', 'category', 'is_junk']


,id,content,category,is_junk
0,1,React组件之间怎么传递数据？props和context应该怎么选择？,React,0
1,2,用useEffect监听窗口大小变化，组件卸载时怎么清除事件？,React,0
2,3,React项目里用了useState但是页面不更新，对象引用没变导致的,React,0
3,4,封装了一个自定义Hook来管理表单状态，支持校验和重置,React,0
4,5,React Router v6的嵌套路由怎么配置？outlet组件放哪里？,React,0


In [21]:
# 对每条帖文执行完整预处理流水线
# clean → jieba分词 → 去停用词 → 同义词归一化 → 空格拼接
df["processed_text"] = df["content"].astype(str).apply(preprocess_to_string)

# 查看预处理效果
sample = df[["content", "processed_text", "category"]].sample(10, random_state=42)
for _, row in sample.iterrows():
    print(f"[{row['category']}]")
    print(f"  原文: {row['content'][:60]}...")
    print(f"  处理: {row['processed_text'][:60]}...")
    print()

[nan]
  原文: 每天早上看一一小朋友咯咯笑着出门，心情无限爽朗。可是办公室里的棘手问题总是让我不知所措...
  处理: 每天 早上 看 一一 小朋友 咯咯 笑 着 出门 心情 无限 爽朗 可是 办公室 里 棘手 总是 不知所措...

[nan]
  原文: 北京起来就变成这样...
  处理: 北京 起来 变成...

[nan]
  原文: 又到了小编送礼物的时间了，先说明一下。之前没送出去的礼物大家不要着急，由于企业那边放假比较早，上班比较晚，现在还没上班。...
  处理: 小编 送礼物 先 说明 之前 没 送出去 礼物 不要 着急 由于 企业 那边 放假 早 上班 晚 现在 没 上班 可怜 ...

[移动端]
  原文: 独立开发者亲测：MLX框架让我的App秒变AI原生！15年iOS老兵的2025新感悟。大家好，我是K哥，一个写了15年i...
  处理: 独立 开发者 亲测 MLX 框架 App 秒 变 AI 原生 15 年 移动端 老兵 2025 新 感悟 好 K 哥 写...

[nan]
  原文: 错别字又出现了，难写成男...
  处理: 错别字 出现 难 写成 男...

[网络配置]
  原文: 请推荐两款能迅速保存图片和文本文档的软件！一我经常上网看文章，看。一 我经常上网看文章，看到好的文章难免想复制下来，但是...
  处理: 请 推荐 两款 能 迅速 保存 图片 文本文档 软件 一 经常 上网 看 看 一 经常 上网 看 好 难免 想 复制 下...

[nan]
  原文: 好办法...这就下单去买小白板。可是男人都像这位童鞋那么自觉吗？...
  处理: 好 办法 ... 下单 去 买 小 白板 可是 男人 像 这位 童鞋 自觉...

[网络]
  原文: 老问题，你们断线厉害吗？至尊翔龙区，最近bb挂机你们断线厉害吗？。至尊翔龙区，最近 b b挂 机你们断线厉害吗？ 20个...
  处理: 老 你们 断线 厉害 至尊 翔龙区 bb 挂机 你们 断线 厉害 至尊 翔龙区 b b 挂 机 你们 断线 厉害 20 ...

[nan]
  原文: 不求最好，但求最贵！1600mm，这王子准备干啥，不会偷拍啊，...
  处理: 求 最好 但求 最贵 1600mm 王子 准备 干 啥 不会 偷拍...

[nan]


In [22]:
# 检查预处理后为空的行（纯表情、纯图片、纯标点等）
empty_mask = df["processed_text"].str.strip() == ""
empty_rows = df[empty_mask]

print(f"预处理后为空的行: {len(empty_rows)} 条")

if len(empty_rows) > 0:
    print("\n空行的原始内容:")
    for _, row in empty_rows.iterrows():
        print(f"  id={row['id']}, category={row['category']}, content={row['content'][:50]}")

    # 剔除空行
    df = df[~empty_mask].reset_index(drop=True)
    print(f"\n剔除后剩余: {len(df)} 条")
else:
    print("没有空行，数据完整。")

预处理后为空的行: 1 条

空行的原始内容:
  id=4512, category=nan, content=如果这样，那么。

剔除后剩余: 7727 条


In [23]:
from sklearn.model_selection import train_test_split
from config import TEST_SIZE, RANDOM_STATE

# 用 category 做分层依据，保证各类别在 train/test 中比例一致
# 对于 is_junk=1 的行，category 为空字符串，也会被当作一个独立层
df["category"] = df["category"].fillna("").astype(str)

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["category"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"训练集: {len(train_df)} 条")
print(f"测试集: {len(test_df)} 条")
print(f"比例:   {len(test_df) / len(df) * 100:.1f}%")

训练集: 6181 条
测试集: 1546 条
比例:   20.0%


In [24]:
from config import TRAIN_FILE, TEST_FILE

# 确保目录存在
os.makedirs(os.path.dirname(str(TRAIN_FILE)), exist_ok=True)

train_df.to_csv(str(TRAIN_FILE), index=False, encoding="utf-8-sig")
test_df.to_csv(str(TEST_FILE), index=False, encoding="utf-8-sig")

print(f"已保存: {TRAIN_FILE}")
print(f"已保存: {TEST_FILE}")

已保存: D:\code-hub\ml-service\data\processed\train.csv
已保存: D:\code-hub\ml-service\data\processed\test.csv


In [25]:
# 对比训练集和测试集的分布
train_counts = train_df["category"].value_counts().rename("train")
test_counts = test_df["category"].value_counts().rename("test")

stats = pd.concat([train_counts, test_counts], axis=1).fillna(0).astype(int)
stats["total"] = stats["train"] + stats["test"]
stats["train_pct"] = (stats["train"] / stats["total"] * 100).round(1)
stats = stats.sort_values("total", ascending=False)

print("各类别样本分布:")
print("=" * 55)
print(f"{'类别':<12} {'训练集':>6} {'测试集':>6} {'合计':>6} {'训练占比':>8}")
print("-" * 55)
for cat, row in stats.iterrows():
    label = cat if cat else "(垃圾)"
    print(f"{label:<12} {row['train']:>6} {row['test']:>6} {row['total']:>6} {row['train_pct']:>7.1f}%")
print("-" * 55)
print(f"{'合计':<12} {stats['train'].sum():>6} {stats['test'].sum():>6} {stats['total'].sum():>6}")

# 垃圾/技术帖比例
print(f"\n技术帖: {(df['is_junk'] == 0).sum()}, 垃圾帖: {(df['is_junk'] == 1).sum()}")

各类别样本分布:
类别              训练集    测试集     合计     训练占比
-------------------------------------------------------
(垃圾)         4140.0 1035.0 5175.0    80.0%
Windows系统     260.0   65.0  325.0    80.0%
网络            233.0   59.0  292.0    79.8%
软件工具          158.0   40.0  198.0    79.8%
硬件            116.0   29.0  145.0    80.0%
DevOps        107.0   27.0  134.0    79.9%
编程基础          102.0   26.0  128.0    79.7%
移动端           102.0   25.0  127.0    80.3%
编程技术           95.0   24.0  119.0    79.8%
AI/ML          90.0   23.0  113.0    79.6%
办公软件           89.0   22.0  111.0    80.2%
数据库            82.0   21.0  103.0    79.6%
网络配置           82.0   20.0  102.0    80.4%
文件管理           72.0   18.0   90.0    80.0%
前端/浏览器         67.0   17.0   84.0    79.8%
网络安全           62.0   15.0   77.0    80.5%
React          49.0   12.0   61.0    80.3%
后端             44.0   11.0   55.0    80.0%
网络协议           37.0    9.0   46.0    80.4%
CSS            28.0    7.0   35.0    80.0%
JavaScript     27.0    7.0   34.